In [ ]:
# USE_GPTOPIC = False  # whether to use TopicGPT or BERTopic for topic modeling
# USE_TRUNCATED_DOCS = True  # whether to use truncated documents or full documents

# if USE_TRUNCATED_DOCS and not USE_GPTOPIC:
#     TOPIC_OBJS_PATH = "truncated_docs_topic_objs.pkl"  # using truncated documents, BERTopic topic objects
# elif not USE_TRUNCATED_DOCS and not USE_GPTOPIC:
#     TOPIC_OBJS_PATH = "topic_objs.pkl"               # using full documents, BERTopic topic objects
# elif not USE_TRUNCATED_DOCS and USE_GPTOPIC:
#     TOPIC_OBJS_PATH = "topic_objs_gptopic.pkl"       # using full documents, TopicGPT topic objects
# else:   
#     raise ValueError("TopicGPT does not support truncated documents.")

In [8]:
import numpy as np
from typing import List, Optional, Any

class Topic:
    def __init__(
        self,
        document_embeddings_hd: np.ndarray,
        document_embeddings_ld: Optional[np.ndarray] = None,
        centroid_hd: Optional[np.ndarray] = None,
        documents: Optional[List[str]] = None,
        topic_name: Optional[str] = None,
        topic_desc: Optional[str] = None
    ):
        assert isinstance(document_embeddings_hd, np.ndarray), "document_embeddings_hd must be a numpy array"
        if centroid_hd is not None:
            assert isinstance(centroid_hd, np.ndarray), "centroid_hd must be a numpy array"
        if documents is not None:
            assert isinstance(documents, list), "documents must be a list of strings"
        self.document_embeddings_hd = document_embeddings_hd
        self.centroid_hd = centroid_hd
        self.documents = documents or []
        self.topic_name = topic_name
        self.topic_desc = topic_desc
        if document_embeddings_ld is not None:
            assert isinstance(document_embeddings_ld, np.ndarray), "document_embeddings_ld must be a numpy array"
            self.document_embeddings_ld = document_embeddings_ld
        else:
            self.document_embeddings_ld = None

In [9]:
# load the topic objects for truncated 
import pickle
TOPIC_OBJS_PATH = "truncated_docs_topic_objs.pkl" 
with open(TOPIC_OBJS_PATH, "rb") as f:
    truncated_topic_objs = pickle.load(f)

# Check the number of topics loaded
print(f"Loaded {len(truncated_topic_objs)} topics.")

# Check the first topic's name and number of documents
print(truncated_topic_objs[0].topic_name)

total_docs = 0
truncated_document_list = []
truncated_document_embedding_list = []
for topic in truncated_topic_objs:
    total_docs += len(topic.documents)
    truncated_document_list.extend(topic.documents)
    truncated_document_embedding_list.extend(topic.document_embeddings_hd)
print(f"Total documents across all topics: {total_docs}")

Loaded 20 topics.
Sports Team Evaluations
Total documents across all topics: 12762


In [10]:
truncated_document_list.__len__()

12762

In [11]:
# load the topic object for chunked docs
import pickle
TOPIC_OBJS_PATH = "topic_objs.pkl" 
with open(TOPIC_OBJS_PATH, "rb") as f:
    chunked_topic_objs = pickle.load(f)

# Check the number of topics loaded
print(f"Loaded {len(chunked_topic_objs)} topics.")

# Check the first topic's name and number of documents
print(chunked_topic_objs[0].topic_name, len(chunked_topic_objs[0].documents))

total_docs = 0
chunked_document_list = []
chuncked_document_embedding_list = []
for topic in chunked_topic_objs:
    total_docs += len(topic.documents)
    chunked_document_list.extend(topic.documents)
    chuncked_document_embedding_list.extend(topic.document_embeddings_hd)
print(f"Total documents across all topics: {total_docs}")

Loaded 20 topics.
NHL Playoffs April 1993 2043
Total documents across all topics: 17693


In [12]:
chunked_document_list.__len__()
chuncked_document_embedding_list.__len__()

17693

In [13]:
for topic in truncated_topic_objs:
    print(f"Topic: {topic.topic_name}, description: {topic.topic_desc}, num documents: {len(topic.documents)}")

Topic: Sports Team Evaluations, description: Sports analysis covering NHL goaltenders' performances and team standings, MLB predictions, and hockey-related discussions like player ratings and fighting in the league., num documents: 1851
Topic: Computer Hardware for Sale, description: Comparison of SCSI and IDE interfaces for drives, including performance, compatibility, and cost considerations for PCs and Macs. Discussions on expansion options and drive transfer rates., num documents: 1911
Topic: Armenian genocide of Muslims, description: Armenian involvement in the alleged genocide of 2.5 million Muslim people, disputed historical accounts, and its impact on political discourse., num documents: 754
Topic: Telephone Line Voltage Monitoring, description: Telephone Line Technical Issues, num documents: 45
Topic: Becoming a NASA astronaut, description: Space Exploration Overview, num documents: 813
Topic: Government Handling of Firearms JADX, description: Controversy surrounding governmen

In [14]:
# # import dataset and filter out empty documents
# from sklearn.datasets import fetch_20newsgroups

# data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
# corpus = data['data']
# topic_names = data.target_names  # e.g., ['alt.atheism', 'comp.graphics', ...]

# filtered_corpus = []
# filtered_labels = []

# for doc, label in zip(corpus, data.target):
#     if doc != "":
#         filtered_corpus.append(doc)
#         filtered_labels.append(label)

# corpus = filtered_corpus
# labels = filtered_labels

In [15]:
# # utility funcitons. 
# def chunk_text(text, tokenizer, max_tokens):
#     """
#     Split text into chunks, each <= max_tokens (by tokens, not words).
#     Returns: a list of text chunks.
#     """
#     tokens = tokenizer.encode(text)
#     chunks = []
#     for i in range(0, len(tokens), max_tokens):
#         chunk_tokens = tokens[i:i+max_tokens]
#         chunk_text = tokenizer.decode(chunk_tokens)
#         chunks.append(chunk_text)
#     return chunks

# def truncate_text(text, tokenizer, max_tokens):
#     tokens = tokenizer.encode(text)[:max_tokens]
#     return tokenizer.decode(tokens)

In [16]:
# Create DataFrame with documents and their embeddings for truncated documents
import pandas as pd
import tiktoken
## constants
# MAX_TOKENS = 1024  # max tokens per chunk
# tokenizer = tiktoken.encoding_for_model("text-embedding-ada-002")

# truncated_docs = []
# for docs in corpus:
#     chunks = truncate_text(docs, tokenizer, MAX_TOKENS)
#     truncated_docs.append(chunks)

# # Pre-calculate embeddings
# from sentence_transformers import SentenceTransformer
# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
# chunked_docs_embeddings = embedding_model.encode(corpus, show_progress_bar=True)

# Make sure both lists are the same length
assert len(chunked_document_list) == len(chuncked_document_embedding_list), "Lists must be of the same length"

chunked_document_dataframe = pd.DataFrame({
    "Chunked_Document": chunked_document_list,
    "Doc_embedding_hd": chuncked_document_embedding_list  # convert numpy arrays to lists for DataFrame
})

chunked_document_dataframe.head()

,Chunked_Document,Doc_embedding_hd
0,\n\nI am sure some bashers of Pens fans are pr...,"[0.0020780046, 0.023450432, 0.024808863, -0.01..."
1,"\n[stuff deleted]\n\nOk, here's the solution t...","[0.0384479, -0.06030391, 0.033409163, -0.05646..."
2,"\n\n\nYeah, it's the second one. And I believ...","[-0.047790147, 0.078570694, -0.01835263, -0.04..."
3,I don't know the exact coverage in the states....,"[-0.10184622, -0.019739276, -0.020205412, -0.0..."
4,\nBe patient. He has a sore shoulder from cras...,"[0.009323514, 0.038848054, -0.05198862, -0.061..."


In [17]:
# Create DataFrame with documents and their embeddings for chunked documents
import pandas as pd
# import tiktoken
## constants
# MAX_TOKENS = 1024  # max tokens per chunk
# tokenizer = tiktoken.encoding_for_model("text-embedding-ada-002")

# truncated_docs = []
# for docs in corpus:
#     chunks = truncate_text(docs, tokenizer, MAX_TOKENS)
#     truncated_docs.append(chunks)

# # Pre-calculate embeddings
# from sentence_transformers import SentenceTransformer
# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
# truncated_docs_embeddings = embedding_model.encode(truncated_docs, show_progress_bar=True)

# Make sure both lists are the same length
assert len(truncated_document_list) == len(truncated_document_embedding_list), "Lists must be of the same length"

truncated_document_dataframe = pd.DataFrame({
    "Truncated_Document": truncated_document_list,
    "Doc_embedding_hd": truncated_document_embedding_list  # convert numpy arrays to lists for DataFrame
})

truncated_document_dataframe.head()

,Truncated_Document,Doc_embedding_hd
0,\n\nI am sure some bashers of Pens fans are pr...,"[0.0020780046, 0.023450432, 0.024808863, -0.01..."
1,"\n[stuff deleted]\n\nOk, here's the solution t...","[0.0384479, -0.06030391, 0.033409163, -0.05646..."
2,"\n\n\nYeah, it's the second one. And I believ...","[-0.047790147, 0.078570694, -0.01835263, -0.04..."
3,I don't know the exact coverage in the states....,"[-0.10184622, -0.019739276, -0.020205412, -0.0..."
4,\nBe patient. He has a sore shoulder from cras...,"[0.009323514, 0.038848054, -0.05198862, -0.061..."


In [18]:
print(chunked_document_dataframe.shape)
print(truncated_document_dataframe.shape)

(17693, 2)
(12762, 2)


# Metric Implementation

In [19]:
import numpy as np
import re
from typing import List, Optional, Any
from sklearn.metrics.pairwise import cosine_similarity

class ADS_Intruder:
    """
    Intruder-based Average Description Similarity (ADS-Intruder) metric for topic models.

    This metric measures the average cosine similarity (scaled to [0, 1]) between the documents in a cluster
    and the descriptions of all other clusters ("intruder" descriptions). Lower values indicate better topic separation,
    as documents in a cluster are less similar to descriptions of other topics.

    - Range: 0 to 1 (lower is better)
    - Use: To evaluate how well-separated the topics are from each other based on their descriptions.
    """

    def __init__(self, n_docs: int = -1, embedder: Optional[Any] = None):
        """
        Args:
            n_docs (int): Number of documents per topic to use (-1 means all).
            embedder: An object with .encode() method for embedding texts.
        """
        self.n_docs = n_docs
        self.embedder = embedder

    def get_info(self) -> dict:
        """
        Get information about the metric.
        """
        info = {
            "metric_name": "Intruder-based Average Description Similarity (ADS-Intruder)",
            "n_docs": self.n_docs,
            "metric_range": "-1 to 1, lower is better",
            "description": "Average cosine similarity between cluster documents and intruder topic descriptions. Lower scores indicate better topic separation.",
        }
        return info

    def score(self, topics: List[Any]) -> float:
        """
        Args:
            topics: List of Topic-like objects, each with:
                - topic_desc (str): topic description
                - document_embeddings_hd (np.ndarray): document embeddings
                - centroid_hd (optional, np.ndarray): cluster centroid
        Returns:
            float: The average intruder ADS score across all topics.
        """
        assert isinstance(topics, (list, tuple)), "topics must be a list or tuple of Topic objects"
        assert self.embedder is not None, "embedder must be provided"

        # Clean the descriptions and embed them
        descriptions = [self.clean_description(getattr(t, "topic_desc", "")) for t in topics]
        topic_desc_embeddings = self.embedder.encode(descriptions, convert_to_numpy=True)

        # Prepare document embeddings for each cluster
        emb_clusters: List[np.ndarray] = []
        for t in topics:
            emb = getattr(t, "document_embeddings_hd", None)
            assert emb is not None, "Each topic must have 'document_embeddings_hd'"
            arr = np.atleast_2d(np.asarray(emb))
            
            # If n_docs > 0, select the most representative documents
            if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
                n_available = arr.shape[0]
                n_select = min(self.n_docs, n_available)
                centroid = getattr(t, "centroid_hd", None)
                if centroid is None:
                    centroid = arr.mean(axis=0)
                else:
                    centroid = np.asarray(centroid).reshape(-1)
                sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
                top_idx = np.argsort(sims)[-n_select:][::-1]
                arr = arr[top_idx]
            emb_clusters.append(arr)

        # Calculate intruder similarities for each cluster
        cluster_scores: List[float] = []
        
        for i, cluster_docs in enumerate(emb_clusters):
            if cluster_docs.size == 0:
                cluster_scores.append(np.nan)
                continue
            
            intruder_similarities: List[float] = []
            
            # Compare with all other clusters' descriptions (intruders)
            for j, intruder_desc_emb in enumerate(topic_desc_embeddings):
                if j == i:  # Skip own description
                    continue
                
                # Calculate similarity between intruder description and cluster documents
                intruder_desc = intruder_desc_emb.reshape(1, -1)
                sims = cosine_similarity(intruder_desc, cluster_docs)  # Shape: (1, n_docs)
                sims = (sims + 1) / 2  # Scale similarity to [0, 1] 
                avg_sim = np.mean(sims)  # Average similarity for this intruder
                intruder_similarities.append(avg_sim)
            
            # Average across all intruder descriptions for this cluster
            if intruder_similarities:
                cluster_scores.append(np.mean(intruder_similarities))
            else:
                cluster_scores.append(np.nan)

        # Final average across all clusters
        if len(cluster_scores) == 0 or np.all(np.isnan(cluster_scores)):
            return np.nan
        
        return round(np.nanmean(cluster_scores), 4)

    @staticmethod
    def clean_description(desc: str) -> str:
        if not desc:
            return ""
        desc = desc.strip().strip('\'"')
        desc = re.sub(r'(\*\*|\*|`)+', '', desc)
        desc = re.sub(r'^#{1,6}\s*', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'^\s*[\-\*\+]\s+', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'^\s*\d+\.\s+', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'\s+', ' ', desc)
        return desc.strip()

In [20]:
import numpy as np
import re
from typing import List, Optional, Any
from sklearn.metrics.pairwise import cosine_similarity

class ADS:
    """
    Average Description Similarity (ADS) metric for topic models.

    This metric measures the average cosine similarity (scaled to [0, 1]) between the embedding of each document
    and the embedding of its assigned topic description. Higher values indicate that documents are more similar
    to their topic description, reflecting better topic coherence.

    - Range: 0 to 1 (higher is better)
    - Use: To evaluate how well topic descriptions represent their assigned documents.
    """

    def __init__(self, n_docs: int = -1, embedder: Optional[Any] = None):
        """
        Args:
            n_docs (int): Number of documents per topic to use (-1 means all).
            embedder: An object with a .get_embeddings(list_of_texts) method returning a dict with 'embeddings' key.
        """
        self.n_docs = n_docs
        self.embedder = embedder

    def get_info(self) -> dict:
        """
        Get information about the metric.
        """
        info = {
            "metric_name": "Average Description Similarity (ADS)",
            "n_docs": self.n_docs,
            "metric_range": "0 to 1, higher is better",
            "description": "The average cosine similarity (scaled to [0,1]) between the embedding of each document and the embedding of its topic description.",
        }
        return info

    def score(self, topics: List[Any]) -> float:
        """
        Args:
            topics: List of Topic-like objects, each with:
                - topic_description (str)
                - document_embeddings_hd (np.ndarray)
                - centroid_hd (optional, np.ndarray)
        Returns:
            float: The average ADS score across all topics.
        """
        assert isinstance(topics, (list, tuple)), "topics must be a list or tuple of Topic objects"
        assert self.embedder is not None, "embedder must be provided"

        # Clean the descriptions
        descriptions = [self.clean_description(getattr(t, "topic_description", "")) for t in topics]

        # Embed the topic descriptions
        # topic_desc_embeddings = self.embedder.get_embeddings(descriptions)["embeddings"]
        topic_desc_embeddings = self.embedder.encode(descriptions, convert_to_numpy=True)

        # Embed the documents in each topic
        emb_clusters: List[np.ndarray] = []
        for t in topics:
            emb = getattr(t, "document_embeddings_hd", None)
            assert emb is not None, "Each topic must have 'document_embeddings_hd'"
            arr = np.atleast_2d(np.asarray(emb))
            # If n_docs > 0, select the most representative documents (top-k by similarity to centroid)
            if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
                n_available = arr.shape[0]
                n_select = min(self.n_docs, n_available)
                centroid = getattr(t, "centroid_hd", None)
                if centroid is None:
                    centroid = arr.mean(axis=0)
                else:
                    centroid = np.asarray(centroid).reshape(-1)
                sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
                top_idx = np.argsort(sims)[-n_select:][::-1]
                arr = arr[top_idx]
            emb_clusters.append(arr)

        similarity_scores: List[float] = []
        for i in range(len(emb_clusters)):
            desc = topic_desc_embeddings[i].reshape(1, -1)
            docs = emb_clusters[i]
            if docs.size == 0:
                similarity_scores.append(np.nan)
                continue
            sims = cosine_similarity(desc, docs)  # Shape: (1, n_docs)
            sims_01 = (sims + 1) / 2  # Now in [0, 1]
            similarity_scores.append(np.nanmean(sims_01))  # Average similarity for the topic

        if len(similarity_scores) == 0 or np.all(np.isnan(similarity_scores)):
            return np.nan
        return round(np.nanmean(similarity_scores), 4)

    @staticmethod
    def clean_description(desc: str) -> str:
        if not desc:
            return ""
        desc = desc.strip().strip('\'"')
        desc = re.sub(r'(\*\*|\*|`)+', '', desc)
        desc = re.sub(r'^#{1,6}\s*', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'^\s*[\-\*\+]\s+', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'^\s*\d+\.\s+', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'\s+', ' ', desc)
        return desc.strip()

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
import warnings
from typing import List, Optional, Any


class ADC:
    """
    Average Document Coherence (ADC) metric for topic models.

    This metric measures the average cosine similarity (scaled to [0, 1]) between the embeddings of documents in a cluster
    and randomly selected "intruder" documents from other clusters. Lower values indicate better topic separation,
    as documents in a cluster are less similar to documents from other clusters.

    - Range: 0 to 1 (lower is better)
    - Use: To evaluate how well-separated the clusters are from each other.
    """

    def __init__(
        self,
        n_docs: int = -1,  # -1 means all docs in the cluster
        n_intruder_docs: int = 1,
    ):
        assert isinstance(n_docs, int), "n_docs must be an integer"
        assert isinstance(n_intruder_docs, int), "n_intruder_docs must be an integer"
        self.n_docs = n_docs
        self.n_intruder_docs = n_intruder_docs

    def score_one_intr_per_cluster(
        self,
        topic_list: List[Topic],
        random_state: Optional[Any] = None,
    ) -> np.ndarray:
        rng = np.random.default_rng(random_state)
        emb_clusters: List[np.ndarray] = []
        for t in topic_list:
            emb = getattr(t, "document_embeddings_hd", None)
            assert isinstance(emb, np.ndarray), "Topic objects must have document_embeddings_hd field populated as numpy array."
            arr = np.atleast_2d(np.asarray(emb))
            if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
                n_available = arr.shape[0]
                if n_available > self.n_docs:
                    centroid = getattr(t, "centroid_hd", None)
                    if centroid is None:
                        centroid = arr.mean(axis=0)
                    else:
                        centroid = np.asarray(centroid).reshape(-1)
                    sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
                    top_idx = np.argsort(sims)[-self.n_docs:][::-1]
                    arr = arr[top_idx]
            emb_clusters.append(arr)

        scores: List[float] = []
        for i, cluster_emb in enumerate(emb_clusters):
            if cluster_emb.size == 0:
                scores.append(np.nan)
                continue
            other = [np.atleast_2d(c) for j, c in enumerate(emb_clusters) if j != i and c.size > 0]
            if len(other) == 0:
                scores.append(np.nan)
                continue
            other_embs = np.vstack(other)
            intr_idx = int(rng.integers(0, other_embs.shape[0]))
            intr_embedding = other_embs[intr_idx]
            sim = cosine_similarity(intr_embedding.reshape(1, -1), cluster_emb)  # (1, n_docs)
            sim = (sim + 1) / 2  # Scale to [0, 1]
            scores.append(float(np.mean(sim)))
        return np.array(scores)

    def score_per_cluster(self, topic_list: List[Topic]) -> dict:
        score_lis: List[np.ndarray] = []
        for _ in range(self.n_intruder_docs):
            score_per_cluster = self.score_one_intr_per_cluster(
                topic_list
            )
            score_lis.append(score_per_cluster)
        res = np.vstack(score_lis).T
        mean_scores = np.mean(res, axis=1)
        ntopics = len(topic_list)
        results: dict = {}
        for k in range(ntopics):
            preview = ""
            t = topic_list[k]
            if getattr(t, "documents", None):
                preview = " - " + str(t.documents[0])[:40].replace("\n", " ").strip()
            label = f"cluster_{k}{preview}"
            results[label] = float(np.round(mean_scores[k], 5))
        return results

    def score(self, topics: List[Topic]) -> float:
        scores = list(self.score_per_cluster(topics).values())
        if all(np.isnan(scores)):
            warnings.warn("ADC: All clusters returned NaN (no valid intruder comparisons possible).")
            return np.nan
        return float(np.nanmean(scores))

# Metric Calculation for our saved Topic Embedding

In [22]:
# # ADC
# adc = ADC(n_docs=-1, n_intruder_docs=100)
# result_chunked_topic_objs = adc.score(chunked_topic_objs)
# result_truncated_topic_objs = adc.score(truncated_topic_objs)
# print("ADC score on chunked data:", result_chunked_topic_objs)
# print("ADC score on truncated data:", result_truncated_topic_objs)

# ## ADS
# # Pre-calculate embeddings
# from sentence_transformers import SentenceTransformer
# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ads = ADS(n_docs=-1, embedder=embedding_model)
# result_chunked_topic_objs = ads.score(chunked_topic_objs)
# result_truncated_topic_objs = ads.score(truncated_topic_objs)
# print("ADS score on chunked data:", result_chunked_topic_objs)
# print("ADS score on truncated data:", result_truncated_topic_objs)

# # ADS Intruder
# # here lower the better .
# ads_intruder = ADS_Intruder(n_docs=-1, embedder=embedding_model)
# result_chunked_topic_objs = ads_intruder.score(chunked_topic_objs)
# result_truncated_topic_objs = ads_intruder.score(truncated_topic_objs)
# print("ads_intruder score on chunked data:", result_chunked_topic_objs)
# print("ads_intruder score on truncated data:", result_truncated_topic_objs)

### Note:

ADS
- **ADS = 1.0** means perfect topic coherence: documents are maximally similar to their own topic description.
- **ADS = 0.5** means moderate coherence: documents are neither highly similar nor dissimilar to their topic description.
- **ADS = 0.0** means poor coherence: documents are not similar to their topic description at all.

ADS_Intruder
- **ADS_Intruder = 0.0** means perfect topic separation: documents in a cluster are not similar to the descriptions of other clusters.
- **ADS_Intruder = 0.5** means moderate separation: documents are neither highly similar nor dissimilar to other clusters' descriptions.
- **ADS_Intruder = 1.0** means poor separation: documents are as similar as possible to other clusters' descriptions (clusters are not distinct).

ADC
- **ADC = 0.0** means perfect topic separation: documents in a cluster are not similar to intruder documents from other clusters.
- **ADC = 0.5** means moderate separation: documents are neither highly similar nor dissimilar to intruder documents.
- **ADC = 1.0** means poor separation: documents are as similar to intruder documents as possible (clusters are not distinct).



In [23]:
import os
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity

# Assumes Topic and ADC classes are already defined in your notebook

import openai

def get_openai_topic_info(
    representative_docs: List[str],
    api_key: str,
    prompt: str,
    prompt_desc: str,
    cache: Dict[Tuple[str, ...], Tuple[str, str]],
    model: str = "gpt-3.5-turbo-16k"
) -> Tuple[str, str]:
    """
    Get topic name and description from OpenAI API, using cache.
    """
    cache_key = tuple(representative_docs)
    if cache_key in cache:
        return cache[cache_key]
    docs_text = "\n".join(representative_docs)
    prompt_filled = prompt.replace("[DOCUMENTS]", docs_text).replace("[KEYWORDS]", "")
    prompt_desc_filled = prompt_desc.replace("[DOCUMENTS]", docs_text).replace("[KEYWORDS]", "")
    client = openai.OpenAI(api_key=api_key)
    try:
        # Topic name
        response_name = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt_filled}],
            max_tokens=32,
            temperature=0.5,
        )
        topic_name = response_name.choices[0].message.content.strip().replace("topic:", "").strip()
    except Exception as e:
        print(f"Error getting topic name from OpenAI: {e}")
        topic_name = "ERROR"
    try:
        # Topic description
        response_desc = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt_desc_filled}],
            max_tokens=64,
            temperature=0.5,
        )
        topic_desc = response_desc.choices[0].message.content.strip().replace("topic:", "").strip()
    except Exception as e:
        print(f"Error getting topic description from OpenAI: {e}")
        topic_desc = "ERROR"
    cache[cache_key] = (topic_name, topic_desc)
    return topic_name, topic_desc

def random_clustering_adc_scores(
    df: pd.DataFrame,
    doc_col: str,
    emb_col: str,
    n_clusters: int = 21,
    n_random_clusterings: int = 10,
    n_repr_docs: int = 3,
    random_seed: int = 42,
    api_key: str = None,
    prompt: str = None,
    prompt_desc: str = None,
    adc_kwargs: dict = None
) -> Tuple[List[float], List[List[Dict[str, Any]]]]:
    """
    Perform N random clusterings, build Topic objects, compute ADC for each, and store per-cluster info.
    Returns:
        adc_scores: List of ADC scores (one per clustering)
        all_cluster_infos: List of per-clustering lists of per-cluster info dicts
    """
    if api_key is None:
        api_key = os.environ.get('OPENAI_API_KEY')
    if adc_kwargs is None:
        adc_kwargs = dict(n_docs=-1, n_intruder_docs=1000)
    rng = np.random.default_rng(random_seed)
    n_docs = len(df)
    adc_scores = []
    all_cluster_infos = []
    cache = dict()
    for clustering_idx in range(n_random_clusterings):
        # Assign random labels in [-1, 0, ..., n_clusters-1]
        labels = rng.integers(-1, n_clusters, size=n_docs)
        df["RandomCluster"] = labels
        cluster_infos = []
        topic_objs = []
        for cluster_id in range(n_clusters):
            if cluster_id == -1:
                continue
            mask = df["RandomCluster"] == cluster_id
            if not mask.any():
                continue
            docs = df.loc[mask, doc_col].tolist()
            embs = np.stack(df.loc[mask, emb_col].values)
            # Select representative docs (up to n_repr_docs, or 1 if only one doc)
            if len(docs) == 1:
                repr_docs = [docs[0]]
            else:
                repr_rng = np.random.default_rng(random_seed + clustering_idx + cluster_id)
                idxs = repr_rng.choice(len(docs), size=min(n_repr_docs, len(docs)), replace=False)
                repr_docs = [docs[i] for i in idxs]
            # Get topic name/desc from OpenAI (with cache)
            topic_name, topic_desc = get_openai_topic_info(
                representative_docs=repr_docs,
                api_key=api_key,
                prompt=prompt,
                prompt_desc=prompt_desc,
                cache=cache
            )
            centroid = embs.mean(axis=0)
            topic_obj = Topic(
                document_embeddings_hd=embs,
                centroid_hd=centroid,
                documents=docs,
                topic_name=topic_name,
                topic_desc=topic_desc
            )
            topic_objs.append(topic_obj)
            cluster_infos.append({
                "cluster_id": cluster_id,
                "topic_name": topic_name,
                "topic_desc": topic_desc,
                "n_docs": len(docs),
                "repr_docs": repr_docs
            })
        # Compute ADC
        adc = ADC(**adc_kwargs)
        adc_score = adc.score(topic_objs)
        adc_scores.append(adc_score)
        all_cluster_infos.append(cluster_infos)
    return adc_scores, all_cluster_infos

In [24]:
import time

def random_clustering_all_metrics(
    df: pd.DataFrame,
    doc_col: str,
    emb_col: str,
    n_clusters: int = 21,
    n_random_clusterings: int = 10,
    n_repr_docs: int = 3,
    random_seed: int = 42,
    api_key: str = None,
    prompt: str = None,
    prompt_desc: str = None,
    adc_kwargs: dict = None,
    ads_kwargs: dict = None,
    ads_intruder_kwargs: dict = None,
    embedder: Any = None
) -> Tuple[
    list, list, list,
    list
]:
    if api_key is None:
        api_key = os.environ.get('OPENAI_API_KEY')
    if adc_kwargs is None:
        adc_kwargs = dict(n_docs=-1, n_intruder_docs=100)
    if ads_kwargs is None:
        ads_kwargs = dict(n_docs=-1, embedder=embedder)
    if ads_intruder_kwargs is None:
        ads_intruder_kwargs = dict(n_docs=-1, embedder=embedder)
    rng = np.random.default_rng(random_seed)
    n_docs = len(df)
    adc_scores = []
    ads_scores = []
    ads_intruder_scores = []
    all_cluster_infos = []
    cache = dict()
    for clustering_idx in range(n_random_clusterings):
        print(f"Starting clustering {clustering_idx+1}/{n_random_clusterings}...")
        start_time = time.time()
        labels = rng.integers(-1, n_clusters, size=n_docs)
        df["RandomCluster"] = labels
        cluster_infos = []
        topic_objs = []
        for cluster_id in range(n_clusters):
            if cluster_id == -1:
                continue
            mask = df["RandomCluster"] == cluster_id
            if not mask.any():
                continue
            docs = df.loc[mask, doc_col].tolist()
            embs = np.stack(df.loc[mask, emb_col].values)
            if len(docs) == 1:
                repr_docs = [docs[0]]
            else:
                repr_rng = np.random.default_rng(random_seed + clustering_idx + cluster_id)
                idxs = repr_rng.choice(len(docs), size=min(n_repr_docs, len(docs)), replace=False)
                repr_docs = [docs[i] for i in idxs]
            try:
                print(f"  Getting topic name/desc for cluster {cluster_id} in clustering {clustering_idx+1}...")
                topic_name, topic_desc = get_openai_topic_info(
                    representative_docs=repr_docs,
                    api_key=api_key,
                    prompt=prompt,
                    prompt_desc=prompt_desc,
                    cache=cache
                )
            except Exception as e:
                print(f"  Error in OpenAI call for cluster {cluster_id} in clustering {clustering_idx+1}: {e}")
                topic_name, topic_desc = "ERROR", "ERROR"
            centroid = embs.mean(axis=0)
            topic_obj = Topic(
                document_embeddings_hd=embs,
                centroid_hd=centroid,
                documents=docs,
                topic_name=topic_name,
                topic_desc=topic_desc
            )
            topic_objs.append(topic_obj)
            cluster_infos.append({
                "cluster_id": cluster_id,
                "topic_name": topic_name,
                "topic_desc": topic_desc,
                "n_docs": len(docs),
                "repr_docs": repr_docs
            })
        print(f"  Calculating metrics for clustering {clustering_idx+1}...")
        adc = ADC(**adc_kwargs)
        ads = ADS(**ads_kwargs)
        ads_intruder = ADS_Intruder(**ads_intruder_kwargs)
        adc_score = ads_score = ads_intruder_score = None
        try:
            adc_score = adc.score(topic_objs)
            ads_score = ads.score(topic_objs)
            ads_intruder_score = ads_intruder.score(topic_objs)
        except Exception as e:
            print(f"  Error in metric calculation for clustering {clustering_idx+1}: {e}")
        adc_scores.append(adc_score)
        ads_scores.append(ads_score)
        ads_intruder_scores.append(ads_intruder_score)
        all_cluster_infos.append(cluster_infos)
        print(f"Finished clustering {clustering_idx+1}/{n_random_clusterings} in {time.time()-start_time:.1f}s")
    print("All clusterings complete.")
    return adc_scores, ads_scores, ads_intruder_scores, all_cluster_infos 

In [25]:
# # Example usage for truncated and chunked document dataframes
# NUMBER_OF_CLUSTERS = 21
# NUMBER_OF_RANDOM_CLUSTERINGS = 3
# NUMBER_OF_REPR_DOCS = 3
# RANDOM_SEED = 42
# # Define your prompts (as in your notebook)
# prompt = """
# I have a topic that contains the following documents:
# [DOCUMENTS]

# Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
# topic: <topic label>
# """

# prompt_topic_desc = """
# I have a topic that contains the following documents:
# [DOCUMENTS]

# Based on the information above, write a concise but informative description of the topic, summarizing its main themes and aspects. Make the description as wholistic as possible i.e., it describes the topic comprehensively. Use 30-35 words. 
# Make sure it is in the following format: 
# topic: <topic description>
# """

# # For truncated documents
# adc_scores_trunc, cluster_infos_trunc = random_clustering_adc_scores(
#     df=truncated_document_dataframe,
#     doc_col="Truncated_Document",
#     emb_col="Doc_embedding_hd",
#     n_clusters=NUMBER_OF_CLUSTERS,
#     n_random_clusterings=NUMBER_OF_RANDOM_CLUSTERINGS,  # For example, 3 random clusterings
#     n_repr_docs=NUMBER_OF_REPR_DOCS,
#     random_seed=RANDOM_SEED,
#     prompt=prompt,
#     prompt_desc=prompt_topic_desc
# )

# # For chunked documents
# adc_scores_chunk, cluster_infos_chunk = random_clustering_adc_scores(
#     df=chunked_document_dataframe,
#     doc_col="Chunked_Document",
#     emb_col="Doc_embedding_hd",
#     n_clusters=NUMBER_OF_CLUSTERS,
#     n_random_clusterings=NUMBER_OF_RANDOM_CLUSTERINGS,
#     n_repr_docs=NUMBER_OF_REPR_DOCS,
#     random_seed=RANDOM_SEED,
#     prompt=prompt,
#     prompt_desc=prompt_topic_desc
# )

# print("Truncated ADC scores:", adc_scores_trunc)
# print("Chunked ADC scores:", adc_scores_chunk)

# # # Example output for one clustering's cluster info
# # import pprint
# # print("Example cluster info for truncated docs (first clustering):")
# # pprint.pprint(cluster_infos_trunc[0])

In [26]:
# np.mean(adc_scores_trunc), np.mean(adc_scores_chunk)

In [27]:
NUMBER_OF_CLUSTERS = 21
NUMBER_OF_RANDOM_CLUSTERINGS = 50
NUMBER_OF_REPR_DOCS = 3
RANDOM_SEED = 42
# Define your prompts (as in your notebook)
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
topic: <topic label>
"""

prompt_topic_desc = """
I have a topic that contains the following documents:
[DOCUMENTS]

Based on the information above, write a concise but informative description of the topic, summarizing its main themes and aspects. Make the description as wholistic as possible i.e., it describes the topic comprehensively. Use 30-35 words. 
Make sure it is in the following format: 
topic: <topic description>
"""

# from sentence_transformers import SentenceTransformer
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# print("Starting all-metrics random clustering for truncated documents...")
# adc_scores_trunc, ads_scores_trunc, ads_intruder_scores_trunc, cluster_infos_trunc = random_clustering_all_metrics(
#     df=truncated_document_dataframe,
#     doc_col="Truncated_Document",
#     emb_col="Doc_embedding_hd",
#     n_clusters=NUMBER_OF_CLUSTERS,
#     n_random_clusterings=NUMBER_OF_RANDOM_CLUSTERINGS,
#     n_repr_docs=NUMBER_OF_REPR_DOCS,
#     random_seed=RANDOM_SEED,
#     prompt=prompt,
#     prompt_desc=prompt_topic_desc,
#     embedder=embedding_model
# )
# print("Finished all-metrics random clustering for truncated documents.")
# print("Truncated ADC scores:", adc_scores_trunc)
# print("Truncated ADS scores:", ads_scores_trunc)
# print("Truncated ADS Intruder scores:", ads_intruder_scores_trunc)

In [28]:
truncated_adc_scores_list = [0.5201314285714286, 0.519954761904762, 0.5198161904761903, 0.5196647619047619, 0.5195742857142857, 0.5198419047619047, 0.5200266666666666, 0.5199233333333333, 0.5198876190476189, 0.5197552380952382, 0.5200438095238095, 0.5196304761904761, 0.5196895238095237, 0.5195419047619045, 0.5195728571428573, 0.5202104761904762, 0.5193966666666666, 0.5198119047619048, 0.519667619047619, 0.5197271428571428, 0.5201109523809524, 0.5197419047619046, 0.5194814285714286, 0.5197757142857143, 0.5198976190476191, 0.5197866666666666, 0.5199119047619047, 0.5197676190476191, 0.5197914285714286, 0.5199661904761905, 0.5198409523809525, 0.5198004761904762, 0.51984, 0.5198371428571428, 0.5199990476190477, 0.5198142857142856, 0.5197700000000001, 0.5196338095238096, 0.5199742857142857, 0.5200504761904762, 0.5197542857142857, 0.5201823809523811, 0.5200752380952381, 0.5198652380952381, 0.5199133333333333, 0.520224761904762, 0.5198561904761905, 0.519837619047619, 0.5199214285714285, 0.5195895238095238]
truncated_ads_score_list = [
    0.5278, 0.5281, 0.5279, 0.528, 0.5279, 0.5281, 0.5278, 0.5281, 0.528, 0.5281,
    0.5283, 0.5279, 0.5279, 0.528, 0.5278, 0.528, 0.5278, 0.5281, 0.5278, 0.528,
    0.5279, 0.5281, 0.528, 0.528, 0.5283, 0.5278, 0.5279, 0.528, 0.5281, 0.5279,
    0.528, 0.5282, 0.5278, 0.528, 0.5281, 0.528, 0.528, 0.528, 0.5281, 0.5278,
    0.5279, 0.5281, 0.5279, 0.5281, 0.528, 0.528, 0.5279, 0.5282, 0.528, 0.528
]
truncated_ads_intruder_list = [
    0.5224, 0.5261, 0.5221, 0.5183, 0.5201, 0.5223, 0.5217, 0.5247, 0.5244, 0.5207,
    0.5202, 0.5216, 0.5249, 0.5222, 0.5242, 0.5207, 0.5228, 0.5263, 0.5248, 0.5251,
    0.5221, 0.5235, 0.5235, 0.5255, 0.5251, 0.5234, 0.5265, 0.5215, 0.5242, 0.5207,
    0.5227, 0.521, 0.522, 0.5201, 0.524, 0.5227, 0.5228, 0.524, 0.52, 0.5226,
    0.5209, 0.5239, 0.5224, 0.5195, 0.5201, 0.5231, 0.52, 0.5232, 0.5234, 0.5198
]

In [29]:
import numpy as np 
def percentage_higher_than(values, threshold):
    values = np.array(values)
    count_higher = np.sum(values > threshold)
    percentage = 100 * count_higher / len(values)
    return percentage

import numpy as np 
def precentage_lower_than(values, threshold):
    values = np.array(values)
    count_lower = np.sum(values < threshold)
    percentage = 100 * count_lower / len(values)
    return percentage

In [30]:
my_ads_score = 0.5135694
percent = percentage_higher_than(truncated_ads_score_list, my_ads_score)
print(f"{percent:.1f}% of values are higher than {my_ads_score}")

my_ads_intruder_score = 0.51089
percent = precentage_lower_than(truncated_ads_intruder_list, my_ads_intruder_score)
print(f"{percent:.1f}% of values are lower than {my_ads_intruder_score}")

my_adc_score = 0.5135669
percent = precentage_lower_than(truncated_adc_scores_list, my_adc_score)
print(f"{percent:.1f}% of values are lower than {my_adc_score}")

100.0% of values are higher than 0.5135694
0.0% of values are lower than 0.51089
0.0% of values are lower than 0.5135669


In [ ]:
print("Starting all-metrics random clustering for chunked documents...")
adc_scores_chunk, ads_scores_chunk, ads_intruder_scores_chunk, cluster_infos_chunk = random_clustering_all_metrics(
    df=chunked_document_dataframe,
    doc_col="Chunked_Document",
    emb_col="Doc_embedding_hd",
    n_clusters=NUMBER_OF_CLUSTERS,
    n_random_clusterings=NUMBER_OF_RANDOM_CLUSTERINGS,
    n_repr_docs=NUMBER_OF_REPR_DOCS,
    random_seed=RANDOM_SEED,
    prompt=prompt,
    prompt_desc=prompt_topic_desc,
    embedder=embedding_model
)
print("Finished all-metrics random clustering for chunked documents.")
print("Chunked ADC scores:", adc_scores_chunk)
print("Chunked ADS scores:", ads_scores_chunk)
print("Chunked ADS Intruder scores:", ads_intruder_scores_chunk)

Starting all-metrics random clustering for chunked documents...
Starting clustering 1/50...
  Getting topic name/desc for cluster 0 in clustering 1...
  Getting topic name/desc for cluster 1 in clustering 1...
  Getting topic name/desc for cluster 2 in clustering 1...
  Getting topic name/desc for cluster 3 in clustering 1...
  Getting topic name/desc for cluster 4 in clustering 1...
  Getting topic name/desc for cluster 5 in clustering 1...
  Getting topic name/desc for cluster 6 in clustering 1...
  Getting topic name/desc for cluster 7 in clustering 1...
  Getting topic name/desc for cluster 8 in clustering 1...
  Getting topic name/desc for cluster 9 in clustering 1...
  Getting topic name/desc for cluster 10 in clustering 1...
  Getting topic name/desc for cluster 11 in clustering 1...
  Getting topic name/desc for cluster 12 in clustering 1...
  Getting topic name/desc for cluster 13 in clustering 1...
  Getting topic name/desc for cluster 14 in clustering 1...
  Getting topic na

In [ ]:
my_ads_score = 0.5131005
percent = percentage_higher_than(ads_scores_chunk, my_ads_score)
print(f"{percent:.1f}% of values are higher than {my_ads_score}")

my_ads_intruder_score = 0.5096
percent = precentage_lower_than(ads_intruder_scores_chunk, my_ads_intruder_score)
print(f"{percent:.1f}% of values are lower than {my_ads_intruder_score}")

my_adc_score = 0.5131005
percent = precentage_lower_than(adc_scores_chunk, my_adc_score)
print(f"{percent:.1f}% of values are lower than {my_adc_score}")